# Практика: Video AE, conditional Video AE и Video VAE

Теперь вместо одной
картинки модель получает короткий клип: один предмет одежды движется по кадру.
Последовательно построим три модели:

1. **Video AE** — детерминированный код клипа;
2. **conditional Video AE** — тот же код и метка класса для декодера;
3. **Video VAE** — распределение латентного кода и генерация новых клипов.

## 1. Импорты и воспроизводимость

In [ ]:
import copy
import json
import random
import time
from dataclasses import dataclass, asdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import imageio.v2 as imageio
from IPython.display import display, Video

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets

SEED = 42

def seed_everything(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything()
torch.set_num_threads(min(4, torch.get_num_threads()))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Устройство:", device)
if device.type == "cuda":
    print(torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)

## 2. Данные: Moving FashionMNIST

Исходное изображение 28×28 помещаем на чёрный фон 40×40. Объект движется
с постоянной скоростью и отражается от границ. Класс одежды сохраняется.

Видеотензор: `[channels, time, height, width]`, батч: `[B,C,T,H,W]`.
Используем 16 кадров; при просмотре 8 FPS это 2 секунды, плеер можно зациклить.
Отдельное изображение имело `[B,1,28,28]`, теперь вход — `[B,1,16,40,40]`.

Сначала разделим **исходные изображения** на train/validation/test, и только
потом создадим траектории. Один и тот же индекс исходного изображения не попадёт в разные
части набора. Траектории фиксированы для честного сравнения моделей.

In [ ]:
@dataclass
class Config:
    latent_dim: int = 32
    num_classes: int = 10
    frames: int = 16
    canvas_size: int = 40
    fps: int = 8
    batch_size: int = 64
    learning_rate: float = 1e-3
    ae_epochs: int = 15
    vae_epochs: int = 25
    beta: float = 1.0
    train_size: int = 6000
    val_size: int = 1000
    test_size: int = 1000
    fast_mode: bool = False

cfg = Config()
if cfg.fast_mode:
    cfg.train_size, cfg.val_size, cfg.test_size = 2000, 400, 400
    cfg.ae_epochs, cfg.vae_epochs = 5, 8
assert cfg.frames >= 4 and cfg.frames % 4 == 0
assert cfg.canvas_size >= 32 and cfg.canvas_size % 8 == 0

DATA = Path("data_video_vae")
OUT = Path("results_video_vae")
OUT.mkdir(exist_ok=True)

datasets.FashionMNIST.mirrors = [
    "https://raw.githubusercontent.com/zalandoresearch/fashion-mnist/master/data/fashion/"
]
full_train = datasets.FashionMNIST(DATA, train=True, download=True)
full_test = datasets.FashionMNIST(DATA, train=False, download=True)
class_names = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
               "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]

permutation = torch.randperm(len(full_train), generator=torch.Generator().manual_seed(SEED))
train_pool, val_pool = permutation[:55000], permutation[55000:]
train_ids = train_pool[:cfg.train_size]
val_ids = val_pool[:cfg.val_size]
test_ids = torch.randperm(len(full_test), generator=torch.Generator().manual_seed(SEED + 1))[:cfg.test_size]
assert not set(train_ids.tolist()) & set(val_ids.tolist())
print(asdict(cfg))

In [ ]:
class MovingFashionMNIST(Dataset):
    '''Одна фиксированная траектория на исходное изображение; хранение в uint8.'''
    def __init__(self, source, indices, frames=16, canvas_size=40, seed=42):
        rng = np.random.default_rng(seed)
        images = source.data[indices].numpy()
        self.labels = source.targets[indices].clone()
        self.clips = np.zeros((len(images), 1, frames, canvas_size, canvas_size), dtype=np.uint8)
        self.positions = np.zeros((len(images), frames, 2), dtype=np.int64)
        limit = canvas_size - images.shape[-1]
        for i, image in enumerate(images):
            start = rng.integers(0, limit + 1, size=2)
            velocity = rng.choice([-2, -1, 1, 2], size=2)
            for t in range(frames):
                q = (start + velocity * t) % (2 * limit)
                x, y = np.where(q <= limit, q, 2 * limit - q)
                self.positions[i, t] = (x, y)
                self.clips[i, 0, t, y:y + 28, x:x + 28] = image

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        # Масштабирование 0..255 -> 0..1 делаем при чтении батча.
        clip = torch.from_numpy(self.clips[index]).float() / 255
        return clip, self.labels[index]

train_dataset = MovingFashionMNIST(full_train, train_ids, cfg.frames, cfg.canvas_size, seed=10)
val_dataset = MovingFashionMNIST(full_train, val_ids, cfg.frames, cfg.canvas_size, seed=20)
test_dataset = MovingFashionMNIST(full_test, test_ids, cfg.frames, cfg.canvas_size, seed=30)

def make_loader(dataset, shuffle=False):
    return DataLoader(dataset, batch_size=cfg.batch_size, shuffle=shuffle, num_workers=0,
                      pin_memory=(device.type == "cuda"),
                      generator=torch.Generator().manual_seed(SEED))

train_loader = make_loader(train_dataset, shuffle=True)
val_loader = make_loader(val_dataset)
test_loader = make_loader(test_dataset)
print(f"Train: {len(train_dataset):,}; validation: {len(val_dataset):,}; test: {len(test_dataset):,}")
print("Батч:", next(iter(train_loader))[0].shape)

In [ ]:
def plot_clips(clips, row_names=None, title="", shown_frames=8):
    '''Каждая строка — целый клип, столбцы — одинаковые моменты времени.'''
    clips = clips.detach().cpu()
    times = np.linspace(0, clips.shape[2] - 1, shown_frames).round().astype(int)
    fig, axes = plt.subplots(len(clips), len(times), figsize=(1.35 * len(times), 1.5 * len(clips)), squeeze=False)
    for row, clip in enumerate(clips):
        for col, t in enumerate(times):
            axes[row, col].imshow(clip[0, t], cmap="gray", vmin=0, vmax=1)
            axes[row, col].set_xticks([])
            axes[row, col].set_yticks([])
            if row == 0:
                axes[row, col].set_title(f"t={t}")
        if row_names:
            axes[row, 0].set_ylabel(row_names[row], rotation=0, labelpad=45, va="center")
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

def save_clip(clip, filename, fps=None):
    # [1,T,H,W] -> [T,H,W,3], кодек нужен только для просмотра.
    frames = (clip.detach().cpu()[0].clamp(0, 1).numpy() * 255).round().astype(np.uint8)
    rgb = np.repeat(frames[..., None], 3, axis=-1)
    imageio.mimwrite(str(filename), rgb, fps=fps or cfg.fps, codec="libx264", macro_block_size=1,
                    pixelformat="yuv420p", ffmpeg_log_level="error")

def show_batch(loader, n=3):
    clips, labels = next(iter(loader))
    plot_clips(clips[:n], [class_names[i] for i in labels[:n].tolist()], "Примеры: движение одного предмета")
    save_clip(clips[0], OUT / "input_example.mp4")
    display(Video(str(OUT / "input_example.mp4"), embed=True, width=240,
                  html_attributes="controls loop"))

show_batch(train_loader)

## 3. Общие свёрточные блоки: что меняется относительно картинок

Conv2d обрабатывает пространство. Conv3d использует окно сразу по времени и
пространству. Stride может уменьшать T отдельно от H/W. Порядок осей важен:
`Conv3d` ожидает `[B,C,T,H,W]`, а не `[B,T,C,H,W]`.

Энкодер при настройках по умолчанию:

```text
[B,1,16,40,40] → [B,16,16,20,20] → [B,32,8,10,10] → [B,64,4,5,5]
```

Декодер восстанавливает оси в обратном порядке.

In [ ]:
class VideoEncoder(nn.Module):
    def __init__(self, frames=16, canvas_size=40):
        super().__init__()
        self.feature_shape = (64, frames // 4, canvas_size // 8, canvas_size // 8)
        self.feature_dim = int(np.prod(self.feature_shape))
        self.net = nn.Sequential(
            nn.Conv3d(1, 16, kernel_size=(3,4,4), stride=(1,2,2), padding=(1,1,1)),
            nn.BatchNorm3d(16),
            nn.LeakyReLU(0.2),
            nn.Conv3d(16, 32, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm3d(32),
            nn.LeakyReLU(0.2),
            nn.Conv3d(32, 64, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm3d(64),
            nn.LeakyReLU(0.2),
            nn.Flatten(),
        )

    def forward(self, clips):
        return self.net(clips)

class VideoDecoder(nn.Module):
    def __init__(self, input_dim, feature_shape):
        super().__init__()
        self.feature_shape = tuple(feature_shape)
        self.project = nn.Linear(input_dim, int(np.prod(feature_shape)))
        self.net = nn.Sequential(
            nn.ConvTranspose3d(64, 32, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm3d(32),
            nn.LeakyReLU(0.2),
            nn.ConvTranspose3d(32, 16, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm3d(16),
            nn.LeakyReLU(0.2),
            nn.ConvTranspose3d(16, 1, kernel_size=(3,4,4), stride=(1,2,2), padding=(1,1,1)),
            nn.Sigmoid(),
        )

    def forward(self, z):
        features = self.project(z).reshape(z.size(0), *self.feature_shape)
        return self.net(features)

probe = VideoEncoder(cfg.frames, cfg.canvas_size)
h = next(iter(train_loader))[0][:2]
for layer in probe.net:
    h = layer(h)
    print(type(layer).__name__, tuple(h.shape))
del probe, h

## 4. Обычный Video AE

Как в image AE: $z=E_\phi(x)$, $\hat x=D_\theta(z)$.
Теперь x — целый клип, а z содержит информацию о внешнем виде **и** движении.
Простая реконструкция не задаёт известное распределение кодов. Поэтому
случайный $z\sim\mathcal N(0,I)$ не обязан давать осмысленное видео.

In [ ]:
class VideoAutoencoder(nn.Module):
    def __init__(self, latent_dim=24, frames=16, canvas_size=40):
        super().__init__()
        self.latent_dim = latent_dim
        self.encoder_features = VideoEncoder(frames, canvas_size)
        self.to_latent = nn.Linear(self.encoder_features.feature_dim, latent_dim)
        self.decoder = VideoDecoder(latent_dim, self.encoder_features.feature_shape)

    def encode(self, clips):
        return self.to_latent(self.encoder_features(clips))

    def decode(self, z):
        return self.decoder(z)

    def forward(self, clips):
        z = self.encode(clips)
        return self.decode(z), z

## 5. Условный Video AE

Декодер получает `[z; onehot(y)]`, где y — класс одежды:
$\hat x=D_\theta([z;\mathrm{onehot}(y)])$.
Условие задаёт тип предмета, а не направление или скорость движения.

In [ ]:
class ConditionalVideoAutoencoder(nn.Module):
    def __init__(self, latent_dim=24, num_classes=10, frames=16, canvas_size=40):
        super().__init__()
        self.latent_dim = latent_dim
        self.num_classes = num_classes
        self.encoder_features = VideoEncoder(frames, canvas_size)
        self.to_latent = nn.Linear(self.encoder_features.feature_dim, latent_dim)
        self.decoder = VideoDecoder(latent_dim + num_classes, self.encoder_features.feature_shape)

    def encode(self, clips):
        return self.to_latent(self.encoder_features(clips))

    def decode(self, z, labels):
        condition = F.one_hot(labels, num_classes=self.num_classes).to(z.dtype)
        return self.decoder(torch.cat([z, condition], dim=1))

    def forward(self, clips, labels):
        z = self.encode(clips)
        return self.decode(z, labels), z

## 6. Video VAE

Энкодер возвращает μ и logvar распределения **целого клипа**:
$q_\phi(z|x)=\mathcal N(\mu(x),\mathrm{diag}(\sigma^2(x)))$.
Репараметризация:
$z=\mu+\exp(\tfrac12\mathrm{logvar})\odot\epsilon$, $\epsilon\sim\mathcal N(0,I)$.

Декодер создаёт все T кадров из одного кода. Кадры связаны общей причиной z
и совместной архитектурой, но это не гарантирует правильную физику движения.

In [ ]:
class VideoVariationalAutoencoder(nn.Module):
    def __init__(self, latent_dim=24, frames=16, canvas_size=40):
        super().__init__()
        self.latent_dim = latent_dim
        self.encoder_features = VideoEncoder(frames, canvas_size)
        self.to_mu = nn.Linear(self.encoder_features.feature_dim, latent_dim)
        self.to_log_var = nn.Linear(self.encoder_features.feature_dim, latent_dim)
        self.decoder = VideoDecoder(latent_dim, self.encoder_features.feature_shape)

    def encode(self, clips):
        features = self.encoder_features(clips)
        return self.to_mu(features), self.to_log_var(features)

    def reparameterize(self, mu, log_var):
        std = torch.exp(0.5 * log_var)
        eps = torch.randn_like(std)
        return mu + std * eps

    def decode(self, z):
        return self.decoder(z)

    def forward(self, clips):
        mu, log_var = self.encode(clips)
        z = self.reparameterize(mu, log_var)
        return self.decode(z), mu, log_var

probe = VideoVariationalAutoencoder(cfg.latent_dim, cfg.frames, cfg.canvas_size).to(device)
x = next(iter(train_loader))[0][:2].to(device)
reconstruction, mu, log_var = probe(x)
print("input:", x.shape, "mu/logvar:", mu.shape, log_var.shape, "output:", reconstruction.shape)
assert reconstruction.shape == x.shape
del probe, x, reconstruction, mu, log_var

## 7. Функции потерь

Для всех моделей reconstruction — сумма BCE по пикселям, усреднённая по
**batch и кадрам**. Так добавление кадров не увеличивает reconstruction
автоматически в T раз и не ослабляет относительное действие KL.
Одному слагаемому reconstruction соответствует кадр 40×40, а не весь клип.
Для VAE добавляем аналитический KL:
$L=L_{rec}+\beta\cdot\frac12\sum_j(\mu_j^2+\exp(\mathrm{logvar}_j)-1-\mathrm{logvar}_j)$.
KL считаем на целый клип. Поэтому при возврате к сумме BCE по всему клипу
эффективный коэффициент KL равен **T×β**.

In [ ]:
def reconstruction_loss(reconstruction, clips):
    return F.binary_cross_entropy(reconstruction, clips, reduction="sum") / (clips.size(0) * clips.size(2))

def vae_loss(reconstruction, clips, mu, log_var, beta=1.0):
    recon = reconstruction_loss(reconstruction, clips)
    kl = -0.5 * (1 + log_var - mu.square() - log_var.exp()).sum(1).mean()
    return recon + beta * kl, recon, kl

## 8. Универсальный цикл обучения


In [ ]:
def run_epoch(model, loader, model_kind, optimizer=None, beta=1.0):
    is_training = optimizer is not None
    model.train(is_training)
    totals = {"loss": 0.0, "reconstruction": 0.0, "kl": 0.0}
    count = 0
    with torch.set_grad_enabled(is_training):
        for clips, labels in loader:
            clips = clips.to(device)
            labels = labels.to(device)
            if is_training:
                optimizer.zero_grad(set_to_none=True)
            if model_kind == "ae":
                reconstruction, _ = model(clips)
                loss = reconstruction_loss(reconstruction, clips)
                recon_value, kl_value = loss, clips.new_zeros(())
            elif model_kind == "cae":
                reconstruction, _ = model(clips, labels)
                loss = reconstruction_loss(reconstruction, clips)
                recon_value, kl_value = loss, clips.new_zeros(())
            elif model_kind == "vae":
                if is_training:
                    reconstruction, mu, log_var = model(clips)
                else:
                    mu, log_var = model.encode(clips)
                    reconstruction = model.decode(mu)
                loss, recon_value, kl_value = vae_loss(reconstruction, clips, mu, log_var, beta)
            else:
                raise ValueError(model_kind)
            if not torch.isfinite(loss):
                raise RuntimeError("Loss содержит NaN/Inf")
            if is_training:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 10.0)
                optimizer.step()
            for key, value in [("loss", loss), ("reconstruction", recon_value), ("kl", kl_value)]:
                totals[key] += value.detach().item() * len(clips)
            count += len(clips)
    return {key: value / count for key, value in totals.items()}

def fit_model(model, model_kind, epochs, learning_rate, beta=1.0):
    seed_everything()
    loader = make_loader(train_dataset, shuffle=True)
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    history, best_state = [], None
    best_val_loss = float("inf")
    initial = run_epoch(model, val_loader, model_kind, beta=beta)
    if device.type == "cuda":
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()
    started = time.perf_counter()
    for epoch in range(1, epochs + 1):
        train_metrics = run_epoch(model, loader, model_kind, optimizer, beta)
        val_metrics = run_epoch(model, val_loader, model_kind, beta=beta)
        history.append({"epoch": epoch, **{f"train_{k}": v for k, v in train_metrics.items()},
                        **{f"val_{k}": v for k, v in val_metrics.items()}})
        if val_metrics["loss"] < best_val_loss:
            best_val_loss = val_metrics["loss"]
            best_state = {name: value.detach().cpu().clone() for name, value in model.state_dict().items()}
        print(f"Epoch {epoch:02d}/{epochs}: train={train_metrics['loss']:.2f}, val={val_metrics['loss']:.2f}")
    model.load_state_dict(best_state)
    if device.type == "cuda":
        torch.cuda.synchronize()
    model.training_info = {"seconds": time.perf_counter() - started, "initial_validation": initial,
                           "best_validation_loss": best_val_loss,
                           "peak_allocated_gib": torch.cuda.max_memory_allocated() / 2**30 if device.type == "cuda" else None,
                           "parameters": sum(p.numel() for p in model.parameters())}
    assert best_val_loss < initial["loss"], "Проверьте настройки: validation не улучшилась"
    return model, pd.DataFrame(history)

def plot_history(history, title):
    fig, axes = plt.subplots(1, 3, figsize=(12, 3))
    for ax, key in zip(axes, ["loss", "reconstruction", "kl"]):
        ax.plot(history["epoch"], history[f"train_{key}"], label="train")
        ax.plot(history["epoch"], history[f"val_{key}"], label="validation (mu)")
        ax.set(xlabel="Эпоха", title=key)
        ax.grid(alpha=0.3)
    axes[0].legend()
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

## 9. Обучение Video AE

Вся цель AE направлена на реконструкцию. Посмотрим, насколько хорошо код
сохраняет внешний вид и изменение положения предмета по времени.

In [ ]:
seed_everything()
ae = VideoAutoencoder(cfg.latent_dim, cfg.frames, cfg.canvas_size)
ae, ae_history = fit_model(ae, "ae", cfg.ae_epochs, cfg.learning_rate)
plot_history(ae_history, "Video AE")
print(ae.training_info)

## 10. Обучение conditional Video AE

Передаём правильную метку класса декодеру.

In [ ]:
seed_everything()
cae = ConditionalVideoAutoencoder(cfg.latent_dim, cfg.num_classes, cfg.frames, cfg.canvas_size)
cae, cae_history = fit_model(cae, "cae", cfg.ae_epochs, cfg.learning_rate)
plot_history(cae_history, "Conditional Video AE")
print(cae.training_info)

## 11. Обучение Video VAE

Теперь вместе с reconstruction смотрим KL.

In [ ]:
seed_everything()
vae = VideoVariationalAutoencoder(cfg.latent_dim, cfg.frames, cfg.canvas_size)
vae, vae_history = fit_model(vae, "vae", cfg.vae_epochs, cfg.learning_rate, beta=cfg.beta)
plot_history(vae_history, f"Video VAE, beta={cfg.beta}")
display(vae_history[["epoch", "train_reconstruction", "train_kl", "val_reconstruction", "val_kl"]].tail())
print(vae.training_info)

## 12. Визуальное сравнение реконструкций

Один и тот же клип восстанавливается всеми тремя моделями. Каждая строка —
последовательность кадров, а не разные объекты. Для VAE используем μ.
Обратите внимание на форму предмета, размытость, направление и момент отражения.

In [ ]:
@torch.no_grad()
def reconstruct(model, clips, labels, kind):
    model.eval()
    if kind == "ae":
        return model(clips)[0]
    if kind == "cae":
        return model(clips, labels)[0]
    mu, _ = model.encode(clips)
    return model.decode(mu)

@torch.no_grad()
def reconstruct_all(n=3):
    clips, labels = next(iter(test_loader))
    clips, labels = clips[:n].to(device), labels[:n].to(device)
    outputs = [clips] + [reconstruct(model, clips, labels, kind)
                         for model, kind in [(ae, "ae"), (cae, "cae"), (vae, "vae")]]
    for i in range(n):
        rows = torch.stack([batch[i] for batch in outputs])
        plot_clips(rows, ["Original", "Video AE", "cVideo AE", "Video VAE"],
                   f"Test: {class_names[labels[i].item()]}")
        combined = torch.cat([batch[i] for batch in outputs], dim=3)
        save_clip(combined, OUT / f"reconstruction_{i}.mp4")
        display(Video(str(OUT / f"reconstruction_{i}.mp4"), embed=True, width=640,
                      html_attributes="controls loop"))

reconstruct_all()

## 13. Численное сравнение: пиксели и время

MSE считаем на один пиксель всего test-набора. Дополнительно сравниваем
временные разности реконструкции и эталона:
`mean(abs((x_hat[t+1]-x_hat[t]) - (x[t+1]-x[t])))`.


In [ ]:
@torch.no_grad()
def evaluate_model(model, loader, kind):
    squared_error, temporal_error = 0.0, 0.0
    pixel_count, difference_count = 0, 0
    for clips, labels in loader:
        clips, labels = clips.to(device), labels.to(device)
        output = reconstruct(model, clips, labels, kind)
        squared_error += F.mse_loss(output, clips, reduction="sum").item()
        difference = (torch.diff(output, dim=2) - torch.diff(clips, dim=2)).abs()
        temporal_error += difference.sum().item()
        pixel_count += clips.numel()
        difference_count += difference.numel()
    return {"test_mse_per_pixel": squared_error / pixel_count,
            "temporal_difference_mae": temporal_error / difference_count}

comparison = pd.DataFrame([
    {"model": name, **evaluate_model(model, test_loader, kind),
     "latent_scalars_per_clip": cfg.latent_dim, **{k: model.training_info[k]
     for k in ["parameters", "seconds", "peak_allocated_gib"]}}
    for model, kind, name in [(ae, "ae", "Video AE"), (cae, "cae", "Conditional Video AE"),
                               (vae, "vae", "Video VAE")]
])
display(comparison)
comparison.to_csv(OUT / "comparison.csv", index=False)

## 14. Проверка влияния условия в cVideo AE

Кодируем один клип один раз и меняем только y. Это аналог опыта из image-практики.
Модель обучалась с правильными классами, поэтому для «чужой» метки не гарантируется
ни идеальная смена типа одежды, ни сохранение траектории.

In [ ]:
@torch.no_grad()
def visualize_condition_effect():
    cae.eval()
    clips, labels = next(iter(test_loader))
    clip = clips[:1].to(device)
    z = cae.encode(clip).repeat(cfg.num_classes, 1)
    outputs = cae.decode(z, torch.arange(cfg.num_classes, device=device))
    plot_clips(outputs, class_names, f"Один z, разные классы; исходный класс: {class_names[labels[0].item()]}", shown_frames=4)
    combined = torch.cat(list(outputs[:5]), dim=3)
    save_clip(combined, OUT / "condition_effect.mp4")
    display(Video(str(OUT / "condition_effect.mp4"), embed=True, width=600,
                  html_attributes="controls loop"))

visualize_condition_effect()

## 15. Генерация новых видео с Video VAE

Выбираем $z\sim\mathcal N(0,I)$ и вызываем декодер **без исходного клипа**.
Один z создаёт всю временную последовательность; новые z — новые клипы.
Показываем средние интенсивности декодера. Векторы prior не обязаны давать
чёткие классы: на небольшом наборе и при ограниченном обучении возможны
размытые силуэты, пропадание объекта и слабое движение.

In [ ]:
@torch.no_grad()
def sample_vae(model, n=8):
    model.eval()
    z = torch.randn(n, model.latent_dim, generator=torch.Generator(device=device).manual_seed(314), device=device)
    samples = model.decode(z).cpu()
    plot_clips(samples, [f"z {i}" for i in range(n)], "Новые клипы: z ~ N(0,I)", shown_frames=6)
    top = torch.cat([samples[0], samples[1]], dim=3)
    bottom = torch.cat([samples[2], samples[3]], dim=3)
    save_clip(torch.cat([top, bottom], dim=2), OUT / "generated_grid.mp4")
    display(Video(str(OUT / "generated_grid.mp4"), embed=True, width=320,
                  html_attributes="controls loop"))
    return samples

generated_samples = sample_vae(vae)

## 16. Интерполяция в латентном пространстве

$z(\alpha)=(1-\alpha)z_1+\alpha z_2$, $\alpha\in[0,1]$.
У Video VAE два независимых направления просмотра:
**по строкам** меняется α, **по столбцам** идёт время внутри каждого клипа.
Для VAE берём μ.

In [ ]:
def select_different_class_pair(loader):
    clips, labels = next(iter(loader))
    second = next(i for i, label in enumerate(labels) if label != labels[0])
    ids = [0, second]
    return clips[ids].to(device), labels[ids].to(device)

def linear_latent_path(start, end, steps=5):
    alpha = torch.linspace(0, 1, steps, device=start.device)[:, None]
    return (1 - alpha) * start + alpha * end

@torch.no_grad()
def interpolate_model(model, kind, steps=5):
    model.eval()
    pair, labels = select_different_class_pair(test_loader)
    codes = model.encode(pair)
    if kind == "vae":
        codes = codes[0]  # mu
    path = linear_latent_path(codes[0], codes[1], steps)
    if kind == "cae":
        outputs = model.decode(path, labels[0].repeat(steps))
    else:
        outputs = model.decode(path)
    plot_clips(outputs, [f"a={a:.2f}" for a in np.linspace(0, 1, steps)],
               f"{kind}: интерполяция клипов; слева/справа по коду — {class_names[labels[0].item()]}/{class_names[labels[1].item()]}",
               shown_frames=6)
    if kind == "vae":
        save_clip(torch.cat(list(outputs), dim=3), OUT / "interpolation_vae.mp4")
        display(Video(str(OUT / "interpolation_vae.mp4"), embed=True, width=600,
                      html_attributes="controls loop"))

interpolate_model(ae, "ae")
interpolate_model(cae, "cae")
interpolate_model(vae, "vae")

## 17. Двумерная проекция латентного пространства

Цвет — класс одежды, каждая точка — **весь клип**, для VAE используем μ.


In [ ]:
@torch.no_grad()
def collect_latents(model, loader, kind, max_objects=1000):
    model.eval()
    latents, labels_all = [], []
    for clips, labels in loader:
        code = model.encode(clips.to(device))
        if kind == "vae":
            code = code[0]
        latents.append(code.cpu())
        labels_all.append(labels)
        if sum(len(x) for x in labels_all) >= max_objects:
            break
    return torch.cat(latents)[:max_objects], torch.cat(labels_all)[:max_objects]

def pca_2d(latents):
    centered = latents - latents.mean(0, keepdim=True)
    _, _, vh = torch.linalg.svd(centered, full_matrices=False)
    return centered @ vh[:2].T

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, model, kind in zip(axes, [ae, cae, vae], ["ae", "cae", "vae"]):
    latents, labels = collect_latents(model, test_loader, kind)
    points = pca_2d(latents)
    scatter = ax.scatter(points[:, 0], points[:, 1], c=labels, cmap="tab10", vmin=-0.5, vmax=9.5, s=8, alpha=0.6)
    ax.set(title=kind, xlabel="PC 1", ylabel="PC 2")
    ax.grid(alpha=0.2)
fig.colorbar(scatter, ax=axes.ravel().tolist(), ticks=range(10), label="Класс", shrink=0.8)
plt.show()

## 18. Расширенный эксперимент с гиперпараметрами

При включении меняем один фактор:
размер bottleneck для Video AE или β для Video VAE. Для каждого запуска
фиксируем данные, инициализацию и число эпох; ранжируем по общим test-метрикам

In [ ]:
RUN_EXTENDED_EXPERIMENTS = False

def run_extended_experiments():
    epochs = 3 if cfg.fast_mode else 10
    rows = []
    for latent_dim in [8, 24, 64]:
        seed_everything()
        model = VideoAutoencoder(latent_dim, cfg.frames, cfg.canvas_size)
        model, _ = fit_model(model, "ae", epochs, cfg.learning_rate)
        rows.append({"kind": "ae", "latent_dim": latent_dim, "beta": 0,
                     **evaluate_model(model, test_loader, "ae")})
        del model
    for beta in [0.1, 1.0, 4.0]:
        seed_everything()
        model = VideoVariationalAutoencoder(cfg.latent_dim, cfg.frames, cfg.canvas_size)
        model, _ = fit_model(model, "vae", epochs, cfg.learning_rate, beta)
        rows.append({"kind": "vae", "latent_dim": cfg.latent_dim, "beta": beta,
                     **evaluate_model(model, test_loader, "vae")})
        del model
    return pd.DataFrame(rows)

if RUN_EXTENDED_EXPERIMENTS:
    extended_comparison = run_extended_experiments()
    display(extended_comparison)
    extended_comparison.to_csv(OUT / "extended_comparison.csv", index=False)

## 19. Сохранение моделей и результатов

Сохраняем state_dict **и конфигурацию**, потому что размер Linear/reshape
зависит от T и H/W. Большие сторонние модели не используются.

In [ ]:
for model, name, history in [(ae, "video_ae", ae_history), (cae, "conditional_video_ae", cae_history),
                              (vae, "video_vae", vae_history)]:
    torch.save({"state_dict": {k: v.detach().cpu() for k, v in model.state_dict().items()},
                "config": asdict(cfg)}, OUT / f"{name}.pt")
    history.to_csv(OUT / f"{name}_history.csv", index=False)
report = {"config": asdict(cfg), "device": str(device), "torch": torch.__version__,
          "gpu": torch.cuda.get_device_name(0) if device.type == "cuda" else None,
          "models": {"ae": ae.training_info, "cae": cae.training_info, "vae": vae.training_info},
          "test_metrics": comparison.to_dict(orient="records"), "seed": SEED,}
(OUT / "run_report.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
print("Модели, кривые, метрики и MP4 сохранены в", OUT.resolve())